In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

<font size="5" color="black">ch6. LSTM기반 Seq2Seq</font>
- 스마트 번역기
 <br> <br>
- GNMT (Google Neural Machine Translation)
- RNN 기반의 Seq2Seq 방식 : 인코더와 디코더가 연결된 구조
    - 인코더 입력
    - 디코더 입력 (모범 답안)
    - 디코더 출력

![Seq2Seq](https://velog.velcdn.com/images/so_oni_on/post/dac0d7cd-4c2d-47b3-8f09-10b33bf38271/image.png)

# 1. pkg 불러오기 및 하이퍼 파라미터 설정

In [2]:
import numpy as np
import pandas as pd
from time import time

from tensorflow.keras.layers import Input, Dense, LSTM
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

In [3]:
MY_HIDDEN = 128
MY_EPOCH = 500

# 2. 학습 데이터셋

In [4]:
raw = pd.read_csv('data/translate.csv')

In [5]:
eng_kor = raw.values.tolist()
print(eng_kor[:3])
print('학습할 데이터 수 :',len(eng_kor))

[['come', '오다'], ['cook', '요리'], ['copy', '복사']]
학습할 데이터 수 : 109


In [6]:
e_alpha = [c for c in 'SEPabcdefghijklmnopqrstuvwxyz']
korean = ''.join([data[1] for data in eng_kor])
k_alpha = list(set([ch for ch in korean]))
k_alpha.sort()

In [7]:
# 영어와 한글 낱말
alpha = e_alpha + k_alpha

In [8]:
# 전체 낱말 개수
alpha_total_size = len(alpha)

# 3. 숫자쌍을 가진 dict 생성
- ex) {'S' : 0, 'E' : 1, 'P' : 2,...}

In [9]:
char_to_num = {}
for i, ch in enumerate(alpha):
    char_to_num[ch] = i

In [10]:
print('문자→숫자 char_to_num.get() : ', char_to_num.get('c'))
print('숫자→문자 alpha[] : ', alpha[5])

문자→숫자 char_to_num.get() :  5
숫자→문자 alpha[] :  c


In [11]:
data = eng_kor[0]
print(data)

# 원-핫 인코딩 전 입력 데이터 형태
print('인코더 입력 :', [char_to_num.get(ch) for ch in data[0]])
print('디코더 입력 :', [char_to_num.get(ch) for ch in 'S' + data[1]])
print('디코더 출력 :', [char_to_num.get(ch) for ch in data[1] + 'E'])

['come', '오다']
인코더 입력 : [5, 17, 15, 7]
디코더 입력 : [0, 118, 59]
디코더 출력 : [118, 59, 1]


In [12]:
# 원-핫 인코딩 1 
to_categorical([5, 17, 15, 7], num_classes=alpha_total_size)[:1]

array([[0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32)

In [13]:
# 원-핫 인코딩 2
np.eye(alpha_total_size)[[5, 17, 15, 7]][:1]

array([[0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

# 4. 인코더/디코더 입력, 디코더 출력
- 인코더 입력과 디코더 입력 (원-핫 인코딩), 디코더 출력

In [37]:
def encoding(eng_kor=eng_kor):
    '인코더 입력 데이터, 디코더 입력 데이터, 디코더 출력 데이터를 return'
    
    enc_in = [] # 인코더 입력
    dec_in = [] # 디코더 입력
    dec_out = [] # 디코더 출력
    
    for data in eng_kor:
        # 인코더 입력 (영어 → 숫자 → 원-핫 인코딩)
        eng = [char_to_num.get(ch) for ch in data[0]]
        eng_one = to_categorical(eng, num_classes=alpha_total_size)
        enc_in.append(eng_one)
        # 디코더 입력 ('S'+ 한글 → 숫자 → 원-핫 인코딩)
        kor = [char_to_num.get(ch) for ch in 'S'+data[1]]
        kor_one = np.eye(alpha_total_size)[kor]
        dec_in.append(kor_one)
        # 디코더 출력 (한글 + 'E' → 숫자)
        kor = [[char_to_num.get(ch)] for ch in data[1]+'E']
        dec_out.append(kor)
    
    # 인공신경망에 넣기 위한 전처리 : numpy 배열로 전환
    enc_in = np.array(enc_in)
    dec_in = np.array(dec_in)
    dec_out = np.array(dec_out)
    
    return enc_in, dec_in, dec_out

In [38]:
sample = [['come','오다']]
enc_in, dec_in, dec_out = encoding(sample)
print(enc_in.shape, dec_in.shape, dec_out.shape)

(1, 4, 170) (1, 3, 170) (1, 3, 1)


# 5. 전체 번역데이터(독립변수, 타겟변수)

In [39]:
# Seq2Seq에 들어갈 데이터
X_enc, X_dec, Y_dec = encoding(eng_kor)
X_enc.shape, X_dec.shape, Y_dec.shape

((109, 4, 170), (109, 3, 170), (109, 3, 1))

# 6. 모델 구현

In [40]:
# 인코더 입력
ENC_IN = Input(shape=(4, alpha_total_size))

# 인코더 LSTM
_, state_h, state_c = LSTM(
    units=MY_HIDDEN,
    return_state=True, # 디코더의 초기 상태로 주입될 h와 c 반환 (기본값 : False)
    # return_sequences=True # True 설정 시 인코더의 모든 시점 출력 (기본값 : False)
)(ENC_IN)

# 인코더와 디코더를 연결할 link
link = [state_h, state_c]

# 디코더 입력
DEC_IN = Input(shape=(3, alpha_total_size))

# 디코더 LSTM
DEC_MID = LSTM(
    units=MY_HIDDEN,
    return_sequences=True
)(DEC_IN, initial_state=link) # 디코더 입력과 디코더 초기 상태 주입을 동시에

# 디코더 출력
DEC_OUT = Dense(units=alpha_total_size, activation='softmax')(DEC_MID)

# 모델 정의
model = Model(
    inputs=[ENC_IN, DEC_IN], 
    outputs=DEC_OUT,
    name='model'
)

model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_5 (InputLayer)           [(None, 4, 170)]     0           []                               
                                                                                                  
 input_6 (InputLayer)           [(None, 3, 170)]     0           []                               
                                                                                                  
 lstm_4 (LSTM)                  [(None, 128),        153088      ['input_5[0][0]']                
                                 (None, 128),                                                     
                                 (None, 128)]                                                     
                                                                                              

# 7. 모델 학습

In [41]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='rmsprop',
    metrics=['accuracy']
)

begin = time()

hist = model.fit(
    [X_enc, X_dec], Y_dec,
    epochs=MY_EPOCH,
    verbose=0
)

end = time()

print(f'학습시간 : {end-begin:.2f}초')

학습시간 : 13.37초


In [33]:
model.evaluate([X_enc, X_dec], Y_dec)

4/4 [==============================] - 0s 4ms/step - loss: 5.8547e-07 - accuracy: 1.0000


[5.854741402799846e-07, 1.0]

# 8. 모델 사용

In [42]:
easy_test = [['down', 'pp'],
             ['desk', 'pp'],
             ['love', 'pp'],
             ['wood', 'pp']]
enc_in, dec_in, _ = encoding(easy_test)
enc_in.shape, dec_in.shape, _.shape

((4, 4, 170), (4, 3, 170), (4, 3, 1))

In [43]:
pred = model.predict([enc_in, dec_in])
pred.argmax(axis=-1)

1/1 [==============================] - 0s 405ms/step


array([[110,  66,   1],
       [148,  96,   1],
       [ 95,  65,   1],
       [ 47,  82,   1]], dtype=int64)

In [44]:
for i in range(len(pred)):
    eng = easy_test[i][0]
    hat = pred[i].argmax(axis=-1)
    kor = ''.join([alpha[num] for num in hat[:-1]])
    print(f'{eng} : {kor}')

down : 아래
desk : 책상
love : 사랑
wood : 나무


In [46]:
# 오타가 있는 단어
hard_test = [['dewn', 'pp'],
             ['desj', 'pp'],
             ['loev', 'pp'],
             ['wodd', 'pp']]
enc_in, dec_in, _ = encoding(easy_test)
enc_in.shape, dec_in.shape, _.shape

((4, 4, 170), (4, 3, 170), (4, 3, 1))

In [47]:
pred = model.predict([enc_in, dec_in])
pred.argmax(axis=-1)

1/1 [==============================] - 0s 13ms/step


array([[110,  66,   1],
       [148,  96,   1],
       [ 95,  65,   1],
       [ 47,  82,   1]], dtype=int64)

In [49]:
for i in range(len(pred)):
    eng = hard_test[i][0]
    hat = pred[i].argmax(axis=-1)
    kor = ''.join([alpha[num] for num in hat[:-1]])
    print(f'{eng} : {kor}')

dewn : 아래
desj : 책상
loev : 사랑
wodd : 나무
